In [105]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [125]:
import pandas as pd
from preprocess import process_description, process_shop_category_name
import polars as pl
from sentence_transformers import SentenceTransformer
import torch
from sklearn.cluster import KMeans


In [112]:
df = pd.read_csv("train.tsv", sep="\t")

In [115]:
df = df.drop(["vendor_name", "vendor_code"], axis=1)
df = process_description(df)

In [37]:
model = SentenceTransformer("cointegrated/rubert-tiny2")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 387.14it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
embeddings = model.encode(df["shop_category_name"].to_list(), convert_to_tensor=True)

In [117]:
embeddings_copy = embeddings.clone()
no_cat_name_mask = (df["shop_category_name"] == "-").to_numpy().astype(bool)
embeddings_copy[no_cat_name_mask] = 0

In [121]:
kmeans = KMeans()
clusters = kmeans.fit_predict(embeddings_copy.cpu().numpy())

In [126]:
top_cats = df["shop_category_name"].value_counts().head(21).index.to_list()

In [131]:
df = process_shop_category_name(df, top_cats, model, kmeans)